# NanoStat-CTV: first LLM baseline (arm B1) on a free Colab T4

**What this measures.** Arm **B1** is an *untrained* instruction model used as the planner. It reads the question and a data profile and writes a
typed analysis plan. NanoStat's deterministic tools run the plan, and the two-gate verifier checks it. No fine-tuning and no RL happen here:
this is the starting point that SFT and GRPO must beat. Metrics come from the answer keys, never from the model.

**Model:** `Qwen/Qwen2.5-Coder-1.5B-Instruct`, the backbone planned in SPEC.md, served with vLLM on `127.0.0.1` inside this Colab VM.
NanoStat's client refuses non-local endpoints, so the synthetic data never leave the VM.

**Rules (SPEC §21, CLAUDE.md):** dev split only here, with no prompt tuning on the locked test split or P-Bench. Report every run, including
failures. The numbers are "measured on Qwen2.5-Coder-1.5B-Instruct, vLLM, prompt <hash>" (printed below).

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Expect about 5 min of setup, then roughly 10–30 min for the dev split (about 100 tasks; not yet timed).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Anandharajan/nanostat-ctv
%cd nanostat-ctv
!git log -1 --format='%h %s'
!pip -q install -e .

## 1 · Serve the model locally with vLLM (OpenAI-compatible API on 127.0.0.1:8000)

In [ ]:
!pip -q install vllm

In [ ]:
import json
import subprocess
import time
import urllib.request

MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SERVED = "qwen2.5-coder-1.5b-instruct"
BASE = "http://127.0.0.1:8000/v1"
log = open("vllm.log", "w")
server = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL,
        "--served-model-name",
        SERVED,
        "--dtype",
        "half",
        "--max-model-len",
        "4096",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--seed",
        "0",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        print(json.load(urllib.request.urlopen(BASE + "/models", timeout=2))["data"][0]["id"], "is being served")
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see vllm.log") from None
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not start in 15 min; see vllm.log")

## 2 · Generate the pilot benchmark (synthetic tasks + answer keys, CPU, about 1–2 min)

In [ ]:
!nanostat factory --profile pilot --out data
!python -c "import hashlib,pathlib; p=pathlib.Path('data/tasks/dev.jsonl'); print('dev tasks', len(p.read_text().splitlines()), 'sha256', hashlib.sha256(p.read_bytes()).hexdigest()[:16])"

## 3 · Smoke test on 5 tasks (checks the whole path before the real run)

In [ ]:
!nanostat eval --arms B1 --llm-model qwen2.5-coder-1.5b-instruct --llm-base-url http://127.0.0.1:8000/v1 \
   --data data --split dev --limit 5 --max-steps 1 --traces results/smoke.jsonl --out results/smoke.json | head -40

## 4 · B1 on the full dev split

Two settings: **single shot** (`--max-steps 1`) measures the raw planner. **Three steps** (`--max-steps 3`) lets the controller re-plan after
verifier feedback, which is how the agent actually runs. Traces are resumable: if Colab disconnects, re-run the cell and it continues.

In [ ]:
!nanostat eval --arms B1 --llm-model qwen2.5-coder-1.5b-instruct --llm-base-url http://127.0.0.1:8000/v1 \
   --data data --split dev --max-steps 1 --traces results/llm_b1/qwen25coder15b_dev_steps1.jsonl \
   --out results/llm_b1/qwen25coder15b_dev_steps1.json > /dev/null
!nanostat eval --arms B1 --llm-model qwen2.5-coder-1.5b-instruct --llm-base-url http://127.0.0.1:8000/v1 \
   --data data --split dev --max-steps 3 --traces results/llm_b1/qwen25coder15b_dev_steps3.jsonl \
   --out results/llm_b1/qwen25coder15b_dev_steps3.json > /dev/null

In [ ]:
import json
import pathlib

for f in sorted(pathlib.Path("results/llm_b1").glob("*.json")):
    r = json.loads(f.read_text())
    keep = {k: v for k, v in r.items() if not isinstance(v, (dict, list))}
    print(f.name)
    for k, v in keep.items():
        print(f"  {k}: {v}")
    for k, v in r.items():
        if isinstance(v, dict) and k in ("metrics", "per_trial"):
            print(f"  {k}: {json.dumps(v)[:600]}")

## 5 · Bring the result back

Download the zip and send it to the maintainer, or commit `results/llm_b1/` in a pull request. The traces contain every raw model output,
the prompt version and hash, verifier errors and the executed results, so anyone can re-score them with `evaluation.llm_eval.score_traces`.

In [ ]:
!cp vllm.log results/llm_b1/ 2>/dev/null; pip freeze | grep -E "^(vllm|torch|transformers)==" > results/llm_b1/versions.txt
!nvidia-smi --query-gpu=name --format=csv,noheader >> results/llm_b1/versions.txt
!zip -qr llm_b1_results.zip results/llm_b1
from google.colab import files

files.download("llm_b1_results.zip")

### Optional: the 7B base model (Fisher-R1's backbone size)
`Qwen/Qwen2.5-Coder-7B-Instruct` does not fit a T4 in fp16. Use the AWQ build: stop the server (`server.terminate()`), restart it with
`--model Qwen/Qwen2.5-Coder-7B-Instruct-AWQ --quantization awq --served-model-name qwen2.5-coder-7b-instruct-awq`, and repeat section 4
with the new model id and output names. Label it "evaluation-matched, not resource-matched" (docs/RELEASE_READINESS.md §3).